In [133]:
# excel 前兩行數據
# MallCode,	OR_Landmark_ID,	MallNameTc,	MallNameEn,	Id,	OR_POI_ID,	ReferenceId,	MallId,	Name_en,	Name_tc,	Name_sc,	CategoryNameEn,	ShopNumber
# tmtplaza,	42,	屯門市廣場,	tmtplaza,	1592,	803423,	540,	1,	&you,	&you,	&you,	Western Cuisine,	3212
# tmtplaza,	42,	屯門市廣場,	tmtplaza,	1605,	794645,	544,	1,	A-1 Bakery Premium,	A-1 Bakery Premium,	A-1 Bakery Premium,	Confectionery & Deli,	UG046

# 我的需求如下
# 1. 讀取Sino POI full list.xlsx的sheet Sino POIs，統計每個landmark的每個orpoiid的以下數據：
# No. of POIs - Booking	
# No. of Booking	
# No. of POIs - Voucher	
# No. of Voucher Transaction	
# No. of POIs - Takeaway	
# No. of Takeaway Transaction	
# Retail POIs Video Views
# 計算邏輯代碼請參考 Sino_poi_stat.sql(請注意，代碼是基於landmark所有商戶統計的，我要的是在excel裏含有的poi：指定商戶)
# 我需要一次性呈現所有結果（像sql一樣），df只能有一個結果，不方便。
# 請保持SET @landmarkId = 37 的設定，我可以自己輸入landmark id生成結果。
# 有些orpoiid是空白的，這種情況請你refer to excel的Id欄，這是mars.dbo.poi的poiid （另外，mars.dbo.poi的orpoiid= openrice3.dbo.poi的poiid
# 有幾個landmarkid是NA，但是有商戶的id(=marspoiid),orpoiid信息，請你refer to excel的MallNameTc(landmark中文名字），cell2是excel例子，請你在所有結果df的顯示當前的landmark名字。
# 請確保輸出的marspoiid是excel的id欄對應的值


In [134]:
# MallCode,	OR_Landmark_ID,	MallNameTc,	MallNameEn,	Id,	OR_POI_ID,	ReferenceId,	MallId,	Name_en,	Name_tc,	Name_sc,	CategoryNameEn,	ShopNumber
# wanchai	N/A	灣仔	Wan Chai	1938	756721		12	Sugidama (The Hennessy)	杉玉居酒屋 (The Hennessy)	杉玉居酒屋 (The Hennessy)	Asian Cuisine	B&C
# wanchai	N/A	灣仔	Wan Chai	1943	31166		12	Wooloomooloo Steakhouse (The Hennessy)	Wooloomooloo Steakhouse (The Hennessy)	Wooloomooloo Steakhouse (The Hennessy)	Western Cuisine	1


In [135]:
import pandas as pd
import pymssql

DB_info = {'server': '192.168.61.119:7622', 'user': 'BAReporting', 'password': 'KeHeCReme8he'}

landmark_id = 42


manual_mall_name_tc = [
]
start_date = '2026-09-01'
end_date = '2026-10-01'

poi_source = pd.read_excel('Sino POI full list.xlsx', sheet_name='Sino POIs')
poi_data = poi_source.rename(columns={
    'OR Landmark ID': 'LandmarkId',
    'OR POI ID': 'ORPoiId',
    'Id': 'MarsPoiId',
}).copy()
poi_data['ORPoiId'] = poi_data['ORPoiId'].astype('string').str.strip().replace('', pd.NA)

def parse_or_poi_ids(value):
    if pd.isna(value):
        return []
    parsed_ids = []
    for value_part in str(value).split('/'):
        poi_id = pd.to_numeric(value_part.strip(), errors='coerce')
        if pd.notna(poi_id):
            parsed_ids.append(int(poi_id))
    return list(dict.fromkeys(parsed_ids))

poi_data['ORPoiIds'] = poi_data['ORPoiId'].map(parse_or_poi_ids)
for column in ['LandmarkId', 'MarsPoiId']:
    poi_data[column] = pd.to_numeric(poi_data[column], errors='coerce').astype('Int64')
for column in ['MallNameTc', 'Name_tc', 'Name_en']:
    poi_data[column] = poi_data[column].astype('string').str.strip().replace('', pd.NA)
poi_data['PoiName'] = poi_data['Name_tc'].fillna(poi_data['Name_en'])

if manual_mall_name_tc:
    selection = poi_data['MallNameTc'].isin(manual_mall_name_tc)
    missing_names = set(manual_mall_name_tc) - set(poi_data['MallNameTc'].dropna())
    if missing_names:
        raise ValueError(f'MallNameTc not found in Excel: {sorted(missing_names)}')
else:
    selection = poi_data['LandmarkId'].eq(landmark_id).fillna(False)

selected_poi_rows = (
    poi_data.loc[selection, ['LandmarkId', 'ORPoiId', 'ORPoiIds', 'MarsPoiId', 'MallNameTc', 'PoiName']]
    .dropna(subset=['ORPoiId', 'MarsPoiId'], how='all')
    .drop_duplicates(subset=['MallNameTc', 'ORPoiId', 'MarsPoiId'])
    .reset_index(drop=True)
)
selected_pois = selected_poi_rows.drop(columns='ORPoiIds')
if selected_pois.empty:
    raise ValueError('No POIs matched the selected landmark_id or manual_mall_name_tc values.')

selected_pois

,LandmarkId,ORPoiId,MarsPoiId,MallNameTc,PoiName
0,42,803423,1592,屯門市廣場,&you
1,42,794645,1605,屯門市廣場,A-1 Bakery Premium
2,42,9527,582,屯門市廣場,東海堂
3,42,858431,2058,屯門市廣場,b.o.t.h. eatery
4,42,690055,2383,屯門市廣場,Bamboo Thai
5,42,536564,1980,屯門市廣場,霸嗎拉麺
6,42,733385,1502,屯門市廣場,荳子
7,42,822773,1962,屯門市廣場,Beans Bakery
8,42,740672,1621,屯門市廣場,BOVE
9,42,1108072,2353,屯門市廣場,Brava Java Coffee


In [136]:
metric_columns = [
    'No. of POIs - Booking',
    'No. of Booking',
    'No. of POIs - Voucher',
    'No. of Voucher Transaction',
    'No. of POIs - Takeaway',
    'No. of Takeaway Transaction',
    'Retail POIs Video Views',
]


def metric_select(scope):
    bbo_scope = f'SELECT BboMarsPoiId FROM #MappedPoi WHERE {scope}'
    video_scope = f'SELECT OpenRicePoiId FROM #MappedPoi WHERE {scope}'
    return f'''
    (SELECT COUNT(DISTINCT booking.PoiId) FROM mars.dbo.Booking AS booking
     WHERE booking.BookingTime > @START AND booking.BookingTime < @END
       AND booking.[Status] = 10 AND booking.PoiId IN ({bbo_scope})),
    (SELECT COUNT_BIG(*) FROM mars.dbo.Booking AS booking
     WHERE booking.BookingTime > @START AND booking.BookingTime < @END
       AND booking.[Status] = 10 AND booking.PoiId IN ({bbo_scope})),
    (SELECT COUNT(DISTINCT poi.NameLang1)
     FROM mars.dbo.Poi AS poi
     INNER JOIN mars.dbo.BizService AS service ON service.PoiId = poi.PoiId
     WHERE service.ServiceStartTime < @END AND service.ServiceEndTime >= @END
       AND service.ServiceTypeId = 5 AND poi.PoiId IN ({bbo_scope})),
    (SELECT COUNT(DISTINCT offer_poi.PoiId)
     FROM mars.dbo.OfferWallet AS wallet
     INNER JOIN mars.dbo.OfferPoi AS offer_poi ON offer_poi.OfferId = wallet.OfferId
     INNER JOIN mars.dbo.Offer AS offer ON offer.OfferId = offer_poi.OfferId
     WHERE wallet.CreateTime > @START AND wallet.CreateTime < @END
       AND wallet.[Status] = 15 AND offer.OfferType IN (7, 18, 22)
       AND offer_poi.PoiId IN ({bbo_scope}) AND wallet.RedeemPoiId IN ({bbo_scope})),
    (SELECT COUNT(DISTINCT takeaway.PoiId)
     FROM mars.dbo.TakeAwayOrder AS takeaway WITH (NOLOCK)
     WHERE takeaway.TakeAwayOrderId >= @MIN_TakeAwayOrderID
       AND takeaway.CreateTime < @END AND takeaway.[Status] = 10
       AND takeaway.PoiId IN ({bbo_scope})),
    (SELECT COUNT_BIG(*) FROM mars.dbo.TakeAwayOrder AS takeaway WITH (NOLOCK)
     WHERE takeaway.TakeAwayOrderId >= @MIN_TakeAwayOrderID
       AND takeaway.CreateTime < @END AND takeaway.[Status] = 10
       AND takeaway.PoiId IN ({bbo_scope})),
    (SELECT COALESCE(SUM(media.HitCount), 0)
     FROM openrice3.dbo.Media AS media
     INNER JOIN openrice3.dbo.Poi AS poi ON poi.PoiId = media.PoiId
     WHERE poi.PoiTypeId = 10 AND media.PoiId IN ({video_scope}))
    '''


sql = f'''
SET NOCOUNT ON;
DECLARE @START datetime = %s;
DECLARE @END datetime = %s;

SELECT DISTINCT
    excel.RowId,
    excel.MallNameTc,
    bbo.PoiId AS BboMarsPoiId,
    COALESCE(excel_or.ORPoiId, bbo.ORPoiId) AS OpenRicePoiId
INTO #MappedPoi
FROM #ExcelPoi AS excel
LEFT JOIN #ExcelPoiORId AS excel_or ON excel_or.RowId = excel.RowId
LEFT JOIN mars.dbo.Poi AS bbo
    ON (excel_or.ORPoiId IS NOT NULL AND excel_or.ORPoiId = bbo.ORPoiId AND bbo.RegionId = 0
        AND EXISTS (SELECT 1 FROM openrice3.dbo.Poi AS poi
                    WHERE poi.PoiId = excel_or.ORPoiId AND poi.[Status] IN (3, 10)))
    OR (excel_or.ORPoiId IS NULL AND excel.MarsPoiId = bbo.PoiId);

DECLARE @MIN_TakeAwayOrderID int;
SELECT @MIN_TakeAwayOrderID = MAX(TakeAwayOrderID)
FROM mars.dbo.TakeAwayOrder WITH (NOLOCK)
WHERE CreateTime < @START;

SELECT excel.RowId, CAST(NULL AS nvarchar(255)) AS SummaryMallNameTc,
       {metric_select('RowId = excel.RowId')}
FROM #ExcelPoi AS excel
UNION ALL
SELECT -ROW_NUMBER() OVER (ORDER BY summary.MallNameTc), summary.MallNameTc,
       {metric_select('MallNameTc = summary.MallNameTc')}
FROM (SELECT DISTINCT MallNameTc FROM #ExcelPoi) AS summary
ORDER BY RowId;
'''

input_rows = [
    (
        row_id,
        int(mars_poi_id) if pd.notna(mars_poi_id) else None,
        str(mall_name) if pd.notna(mall_name) else None,
    )
    for row_id, (mars_poi_id, mall_name) in enumerate(
        selected_pois[['MarsPoiId', 'MallNameTc']].itertuples(index=False, name=None)
    )
]
input_or_rows = [
    (row_id, or_poi_id)
    for row_id, poi_ids in enumerate(selected_poi_rows['ORPoiIds'])
    for or_poi_id in poi_ids
]
with pymssql.connect(**DB_info) as conn:
    with conn.cursor() as cursor:
        cursor.execute(
            'CREATE TABLE #ExcelPoi '
            '(RowId int PRIMARY KEY, MarsPoiId int NULL, MallNameTc nvarchar(255) NULL);'
        )
        cursor.executemany('INSERT INTO #ExcelPoi VALUES (%s, %s, %s)', input_rows)
        cursor.execute(
            'CREATE TABLE #ExcelPoiORId (RowId int NOT NULL, ORPoiId int NOT NULL, '
            'PRIMARY KEY (RowId, ORPoiId));'
        )
        if input_or_rows:
            cursor.executemany(
                'INSERT INTO #ExcelPoiORId (RowId, ORPoiId) VALUES (%s, %s)',
                input_or_rows,
            )
        cursor.execute(sql, (start_date, end_date))
        metrics = pd.DataFrame(
            cursor.fetchall(),
            columns=['RowId', 'SummaryMallNameTc'] + metric_columns,
        )

poi_report = selected_pois.copy()
poi_report.insert(0, 'RowId', range(len(poi_report)))
poi_report = poi_report.merge(
    metrics.loc[metrics['RowId'].ge(0), ['RowId'] + metric_columns],
    on='RowId',
    validate='one_to_one',
)
poi_report = poi_report.drop(columns='RowId')
poi_report.insert(0, 'LandmarkNameTc', poi_report['MallNameTc'])

summary = metrics.loc[metrics['RowId'].lt(0)].copy()
summary['MallNameTc'] = summary['SummaryMallNameTc']
summary['LandmarkNameTc'] = summary['SummaryMallNameTc']
summary['PoiName'] = 'COUNT ALL'
summary['LandmarkId'] = pd.NA
summary['ORPoiId'] = pd.NA
summary['MarsPoiId'] = pd.NA
summary = summary.drop(columns=['RowId', 'SummaryMallNameTc'])
report = pd.concat([poi_report, summary[poi_report.columns]], ignore_index=True)
report = report.astype({'LandmarkId': 'Int64', 'MarsPoiId': 'Int64'})

assert len(poi_report) == len(selected_pois)
assert poi_report['MarsPoiId'].equals(selected_pois['MarsPoiId'])
assert poi_report['ORPoiId'].equals(selected_pois['ORPoiId'])
assert poi_report['PoiName'].equals(selected_pois['PoiName'])
assert len(summary) == selected_pois['MallNameTc'].nunique(dropna=False)
print(f'POI data count all: {len(poi_report)}; summary rows: {len(summary)}')

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
report

POI data count all: 91; summary rows: 1


,LandmarkNameTc,LandmarkId,ORPoiId,MarsPoiId,MallNameTc,PoiName,No. of POIs - Booking,No. of Booking,No. of POIs - Voucher,No. of Voucher Transaction,No. of POIs - Takeaway,No. of Takeaway Transaction,Retail POIs Video Views
0,屯門市廣場,42,803423,1592,屯門市廣場,&you,0,0,1,0,0,0,0
1,屯門市廣場,42,794645,1605,屯門市廣場,A-1 Bakery Premium,0,0,0,0,0,0,0
2,屯門市廣場,42,9527,582,屯門市廣場,東海堂,0,0,1,0,0,0,0
3,屯門市廣場,42,858431,2058,屯門市廣場,b.o.t.h. eatery,1,221,0,0,1,1,0
4,屯門市廣場,42,690055,2383,屯門市廣場,Bamboo Thai,1,124,1,0,1,5,0
5,屯門市廣場,42,536564,1980,屯門市廣場,霸嗎拉麺,1,8,0,0,1,9,0
6,屯門市廣場,42,733385,1502,屯門市廣場,荳子,0,0,1,0,0,0,0
7,屯門市廣場,42,822773,1962,屯門市廣場,Beans Bakery,0,0,0,0,0,0,0
8,屯門市廣場,42,740672,1621,屯門市廣場,BOVE,0,0,0,0,0,0,0
9,屯門市廣場,42,1108072,2353,屯門市廣場,Brava Java Coffee,0,0,0,0,1,4,0


In [137]:
# extreme case 
# MallCode,	OR_Landmark_ID,	MallNameTc,	MallNameEn,	Id,	OR_POI_ID,	ReferenceId,	MallId,	Name_en,	Name_tc,	Name_sc,	CategoryNameEn,	ShopNumber
# citywalk,	65,	荃新天地,	Citywalk,	1430,	714892 / 711644,	287,	2,	Cha Thai / Chicken Factory,	茶泰 / Chicken Factory,	茶泰 / Chicken Factory,	Asian Cuisine,	UG81		


In [138]:
poi_input_rows = [
    (
        row_id,
        int(mars_poi_id) if pd.notna(mars_poi_id) else None,
        str(or_poi_id) if pd.notna(or_poi_id) else None,
        str(mall_name) if pd.notna(mall_name) else None,
        str(poi_name) if pd.notna(poi_name) else None,
    )
    for row_id, (mars_poi_id, or_poi_id, mall_name, poi_name) in enumerate(
        selected_pois[['MarsPoiId', 'ORPoiId', 'MallNameTc', 'PoiName']].itertuples(index=False, name=None)
    )
]
poi_or_id_rows = [
    (row_id, or_poi_id)
    for row_id, poi_ids in enumerate(selected_poi_rows['ORPoiIds'])
    for or_poi_id in poi_ids
]

list_sql = '''
SET NOCOUNT ON;
DECLARE @START datetime = %s;
DECLARE @END datetime = %s;

CREATE TABLE #MappedPoi (
    RowId int NOT NULL,
    ORPoiIdOriginal nvarchar(100) NULL,
    ExcelMarsPoiId int NULL,
    MallNameTc nvarchar(255) NULL,
    PoiName nvarchar(255) NULL,
    BboMarsPoiId int NULL,
    OpenRicePoiId int NULL
);

INSERT INTO #MappedPoi (RowId, ORPoiIdOriginal, ExcelMarsPoiId, MallNameTc, PoiName, BboMarsPoiId, OpenRicePoiId)
SELECT DISTINCT
    excel.RowId,
    excel.ORPoiIdOriginal,
    excel.MarsPoiId,
    excel.MallNameTc,
    excel.PoiName,
    bbo.PoiId,
    COALESCE(excel_or.ORPoiId, bbo.ORPoiId)
FROM #ExcelPoi AS excel
LEFT JOIN #ExcelPoiORId AS excel_or ON excel_or.RowId = excel.RowId
LEFT JOIN mars.dbo.Poi AS bbo
    ON (excel_or.ORPoiId IS NOT NULL AND excel_or.ORPoiId = bbo.ORPoiId AND bbo.RegionId = 0)
    OR (excel_or.ORPoiId IS NULL AND excel.MarsPoiId = bbo.PoiId);

DECLARE @MIN_TakeAwayOrderID int;
SELECT @MIN_TakeAwayOrderID = MAX(TakeAwayOrderId)
FROM mars.dbo.TakeAwayOrder WITH (NOLOCK)
WHERE CreateTime < @START;

;WITH PoiLists AS (
    SELECT DISTINCT
        Category = CAST('Booking' AS nvarchar(30)),
        mapped.RowId,
        mapped.MallNameTc,
        mapped.ExcelMarsPoiId AS MarsPoiId,
        mapped.ORPoiIdOriginal AS ORPoiId,
        mapped.PoiName,
        Views = CAST(NULL AS bigint)
    FROM #MappedPoi AS mapped
    INNER JOIN mars.dbo.Booking AS booking ON booking.PoiId = mapped.BboMarsPoiId
    INNER JOIN mars.dbo.BizService AS service ON service.PoiId = mapped.BboMarsPoiId
    WHERE booking.BookingTime > @START
      AND booking.BookingTime < @END
      AND booking.[Status] = 10

    UNION ALL

    SELECT DISTINCT
        CAST('Voucher' AS nvarchar(30)),
        mapped.RowId,
        mapped.MallNameTc,
        mapped.ExcelMarsPoiId,
        mapped.ORPoiIdOriginal,
        mapped.PoiName,
        CAST(NULL AS bigint)
    FROM #MappedPoi AS mapped
    INNER JOIN mars.dbo.Poi AS poi ON poi.PoiId = mapped.BboMarsPoiId
    INNER JOIN mars.dbo.BizService AS service ON service.PoiId = poi.PoiId
    WHERE service.ServiceStartTime < @END
      AND service.ServiceEndTime >= @END
      AND service.ServiceTypeId = 5

    UNION ALL

    SELECT DISTINCT
        CAST('TAS' AS nvarchar(30)),
        mapped.RowId,
        mapped.MallNameTc,
        mapped.ExcelMarsPoiId,
        mapped.ORPoiIdOriginal,
        mapped.PoiName,
        CAST(NULL AS bigint)
    FROM #MappedPoi AS mapped
    INNER JOIN mars.dbo.Poi AS poi ON poi.PoiId = mapped.BboMarsPoiId
    INNER JOIN mars.dbo.BizService AS service ON service.PoiId = poi.PoiId
    INNER JOIN mars.dbo.TakeAwayOrder AS takeaway WITH (NOLOCK) ON takeaway.PoiId = poi.PoiId
    WHERE service.ServiceStartTime < @END
      AND service.ServiceEndTime >= @END
      AND service.ServiceTypeId = 4
      AND takeaway.TakeAwayOrderId >= @MIN_TakeAwayOrderID
      AND takeaway.CreateTime < @END
      AND takeaway.[Status] = 10

    UNION ALL

    SELECT
        CAST('Retail Video Views' AS nvarchar(30)),
        mapped.RowId,
        mapped.MallNameTc,
        mapped.ExcelMarsPoiId,
        mapped.ORPoiIdOriginal,
        mapped.PoiName,
        COALESCE(SUM(media.HitCount), 0)
    FROM (
        SELECT DISTINCT RowId, MallNameTc, ExcelMarsPoiId, ORPoiIdOriginal, PoiName, OpenRicePoiId
        FROM #MappedPoi
        WHERE OpenRicePoiId IS NOT NULL
    ) AS mapped
    INNER JOIN openrice3.dbo.Poi AS poi ON poi.PoiId = mapped.OpenRicePoiId
    LEFT JOIN openrice3.dbo.Media AS media ON media.PoiId = poi.PoiId
    WHERE poi.PoiTypeId = 10
    GROUP BY mapped.RowId, mapped.MallNameTc, mapped.ExcelMarsPoiId,
             mapped.ORPoiIdOriginal, mapped.PoiName
)
SELECT
    Category,
    MallNameTc AS LandmarkNameTc,
    MarsPoiId,
    ORPoiId,
    PoiName,
    Views
FROM PoiLists
ORDER BY Category, MallNameTc, PoiName, RowId;
'''

with pymssql.connect(**DB_info) as conn:
    with conn.cursor() as cursor:
        cursor.execute(
            'CREATE TABLE #ExcelPoi '
            '(RowId int NOT NULL PRIMARY KEY, MarsPoiId int NULL, ORPoiIdOriginal nvarchar(100) NULL, '
            'MallNameTc nvarchar(255) NULL, PoiName nvarchar(255) NULL);'
        )
        cursor.executemany(
            'INSERT INTO #ExcelPoi (RowId, MarsPoiId, ORPoiIdOriginal, MallNameTc, PoiName) '
            'VALUES (%s, %s, %s, %s, %s)',
            poi_input_rows,
        )
        cursor.execute(
            'CREATE TABLE #ExcelPoiORId (RowId int NOT NULL, ORPoiId int NOT NULL, '
            'PRIMARY KEY (RowId, ORPoiId));'
        )
        if poi_or_id_rows:
            cursor.executemany(
                'INSERT INTO #ExcelPoiORId (RowId, ORPoiId) VALUES (%s, %s)',
                poi_or_id_rows,
            )
        cursor.execute(list_sql, (start_date, end_date))
        poi_list_df = pd.DataFrame.from_records(
            cursor.fetchall(),
            columns=[column[0] for column in cursor.description],
        )

poi_list_df = poi_list_df.astype({'MarsPoiId': 'Int64'})
booking_list = poi_list_df.loc[poi_list_df['Category'].eq('Booking')].reset_index(drop=True)
voucher_list = poi_list_df.loc[poi_list_df['Category'].eq('Voucher')].reset_index(drop=True)
takeaway_list = poi_list_df.loc[poi_list_df['Category'].eq('TAS')].reset_index(drop=True)
retail_video_views_list = poi_list_df.loc[
    poi_list_df['Category'].eq('Retail Video Views')
].reset_index(drop=True)

poi_list_df

,Category,LandmarkNameTc,MarsPoiId,ORPoiId,PoiName,Views
0,Booking,屯門市廣場,2058,858431,b.o.t.h. eatery,None
1,Booking,屯門市廣場,2383,690055,Bamboo Thai,None
2,Booking,屯門市廣場,<NA>,713537,California Pizza Kitchen,None
3,Booking,屯門市廣場,1643,739520,佐賀燒肉谷,None
4,Booking,屯門市廣場,2203,888192,四季椰林海南椰子雞,None
5,Booking,屯門市廣場,1644,739527,山見台式火鍋,None
6,Booking,屯門市廣場,1185,669476,牛気,None
7,Booking,屯門市廣場,2094,774224,稻成,None
8,Booking,屯門市廣場,2430,1129590,綠意坊,None
9,Booking,屯門市廣場,754,25865,翡翠拉麵小籠包,None
